In [1]:
import json
import os
from pathlib import Path
from pprint import pprint

from datasets import load_from_disk
from transformers import AutoModel, AutoTokenizer

from tqdm import tqdm
from colorama import Fore, Style

In [2]:
# for type annotation
# print(type(tokenizer))
# print(type(model))
from transformers.models.bert.tokenization_bert import BertTokenizer
from transformers.models.distilbert.modeling_distilbert import DistilBertModel
from datasets import DatasetDict

In [3]:
# model and tokenizer
model_path = "distilbert/distilbert-base-uncased"
model:DistilBertModel = AutoModel.from_pretrained(model_path)
tokenizer: BertTokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# dataset
data_dir = Path(os.getcwd()).parent/"data"
dataset: DatasetDict = load_from_disk(data_dir/"updated_ai4privacy_300k_pii") # type:ignore
train_ds, val_ds = dataset["train"], dataset["validation"]
print(f"dataset column names: {train_ds.column_names}\n\n")

dataset column names: ['source_text', 'privacy_mask']




In [5]:
print(type(dataset))

<class 'datasets.dataset_dict.DatasetDict'>


In [17]:
# label info
with open(data_dir/"label_info.json") as f:
    label_info: dict = json.load(f)
    
pprint(list(label_info.keys()))

['bio_tags',
 'entity_classes',
 'label2id',
 'id2label',
 'train_bio_counts',
 'description']


In [7]:
example = train_ds[0]
print("dataset example:")
for item in example.items():
    print(item)

dataset example:
('source_text', 'Subject: Group Messaging for Admissions Process\n\nGood morning, everyone,\n\nI hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:\n\n- wynqvrh053 - Meeting at 10:20am\n- luka.burg - Meeting at 21\n- qahil.wittauer - Meeting at quarter past 13\n- gholamhossein.ruschke - Meeting at 9:47 PM\n- pdmjrsyoz1460 ')
('privacy_mask', [{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}, {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}, {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}, {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}, {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}, {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}, {'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'la

In [8]:
ex_tokenized_with_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)
for item in ex_tokenized_with_offsets.items():
    print(item)

('input_ids', [3395, 1024, 2177, 24732, 2005, 20247, 2832, 2204, 2851, 1010, 3071, 1010, 1045, 3246, 2023, 4471, 4858, 2017, 2092, 1012, 2004, 2057, 3613, 2256, 20247, 6194, 1010, 1045, 2052, 2066, 2000, 10651, 2017, 2006, 1996, 6745, 8973, 1998, 3145, 2592, 1012, 3531, 2424, 2917, 1996, 17060, 2005, 2256, 9046, 6295, 1024, 1011, 1059, 6038, 4160, 19716, 2232, 2692, 22275, 1011, 3116, 2012, 2184, 1024, 2322, 3286, 1011, 11320, 2912, 1012, 20934, 10623, 1011, 3116, 2012, 2538, 1011, 1053, 4430, 4014, 1012, 15966, 2696, 13094, 1011, 3116, 2012, 4284, 2627, 2410, 1011, 1043, 14854, 3286, 15006, 20240, 2078, 1012, 22949, 2818, 3489, 1011, 3116, 2012, 1023, 1024, 4700, 7610, 1011, 22851, 2213, 3501, 2869, 7677, 2480, 16932, 16086])
('token_type_ids', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [9]:
tokens = tokenizer.convert_ids_to_tokens(ex_tokenized_with_offsets["input_ids"])
print(tokens)

['subject', ':', 'group', 'messaging', 'for', 'admissions', 'process', 'good', 'morning', ',', 'everyone', ',', 'i', 'hope', 'this', 'message', 'finds', 'you', 'well', '.', 'as', 'we', 'continue', 'our', 'admissions', 'processes', ',', 'i', 'would', 'like', 'to', 'update', 'you', 'on', 'the', 'latest', 'developments', 'and', 'key', 'information', '.', 'please', 'find', 'below', 'the', 'timeline', 'for', 'our', 'upcoming', 'meetings', ':', '-', 'w', '##yn', '##q', '##vr', '##h', '##0', '##53', '-', 'meeting', 'at', '10', ':', '20', '##am', '-', 'lu', '##ka', '.', 'bu', '##rg', '-', 'meeting', 'at', '21', '-', 'q', '##ah', '##il', '.', 'wit', '##ta', '##uer', '-', 'meeting', 'at', 'quarter', 'past', '13', '-', 'g', '##hol', '##am', '##hos', '##sei', '##n', '.', 'rus', '##ch', '##ke', '-', 'meeting', 'at', '9', ':', '47', 'pm', '-', 'pd', '##m', '##j', '##rs', '##yo', '##z', '##14', '##60']


In [10]:
word_ids = ex_tokenized_with_offsets.word_ids()
print(word_ids)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 52, 52, 52, 52, 52, 52, 53, 54, 55, 56, 57, 58, 58, 59, 60, 60, 61, 62, 62, 63, 64, 65, 66, 67, 68, 68, 68, 69, 70, 70, 70, 71, 72, 73, 74, 75, 76, 77, 78, 78, 78, 78, 78, 78, 79, 80, 80, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 89, 89, 89, 89, 89, 89, 89]


In [ ]:
def align_tokens_to_labels(example: dict, token_offsets: list, verbose: bool=False
                           ) -> dict[tuple[int, int],list[tuple[int, int]]]:
    """aligns dataset example privacy masks with their corresponding tokens

    Args:
        example (dict): custom pii dataset example
        token_offsets (list): offsets mapping of tokens
        verbose (bool, optional): print aligned masks. Defaults to False.

    Returns:
        dict[tuple[int, int],list[tuple[int, int]]]: tokens mapped to their privacy mask
    """
    aligned_tokens = {}
    for label in example["privacy_mask"]:
        # store token offsets that fall within start and end of privacy mask
        tokens_in_label = [offset for offset in token_offsets
                           if offset[0] >= label["start"] and offset[1] < label["end"]
        ]
        # map aligned tokens to their label
        aligned_tokens[(label["start"], label["end"])] = tokens_in_label
        if verbose:
            if tokens_in_label:
                print(Fore.GREEN + f"mask value: {label["value"]} -> {len(tokens_in_label)} tokens")
                print(Style.RESET_ALL + "offsets:")
                pprint(tokens_in_label, indent=4, width=15)
            else:
                print(Fore.RED + f"mask value: {label["value"]} didn't align with any tokens")
    
    return aligned_tokens

In [12]:
def get_ner_labels(batch: dict[str, list]) -> dict[str, list[list[str]]]:
    token_offsets = tokenizer(
        batch["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=False,
    )["offset_mapping"]
    
    batch_ner_labels: list[list[str]] = []
    for i, row_masks in enumerate(batch["privacy_mask"]):
        row_ner_labels = []
        for offset in token_offsets[i]:
            # if add_special_tokens is true skip over special tokens (offset with (0,0))
            if offset == (0, 0): 
                row_ner_labels.append("O")
                continue
            # label is "O" unless privacy label is found
            label = "O" 
            # loop through masks to find label
            for privacy_mask in row_masks:
                # end tag is exclusive
                if offset[0] >= privacy_mask["start"] and offset[1] < privacy_mask["end"]:
                    label = privacy_mask["label"]
                    if offset[0] == privacy_mask["start"]:
                        label = "B-" + label
                    else:
                        label = "I-" + label
                    
                    break # break out of for loop when/if label is found
            row_ner_labels.append(label)
            
        batch_ner_labels.append(row_ner_labels)
    
    return {"ner_tags": batch_ner_labels}

In [13]:
dataset = dataset.map(get_ner_labels, batched=True, batch_size=20_000)
print(dataset["train"].features)

Map:   0%|          | 0/177677 [00:00<?, ? examples/s]

Map:   0%|          | 0/47728 [00:00<?, ? examples/s]

{'source_text': Value('string'), 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}), 'ner_tags': List(Value('string'))}


In [79]:
pprint(dataset["train"][0])

{'attention_mask': [1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
           

In [14]:
# get the labels feature from the dataset
train_ner_tags = dataset["train"]["ner_tags"] 
print(next(iter(train_ner_tags)))

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'O', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O', 'O', 'B-TIME', 'I-TIME', 'I-TIME', 'O', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O']


In [ ]:
from collections import Counter
# count every label.
entity_counter = Counter()
with tqdm(train_ner_tags) as pbar:
    for label in pbar:
        entity_counter.update(label)

100%|██████████| 177677/177677 [00:11<00:00, 15130.94it/s]


In [ ]:
print(f"num_labels = {len(entity_counter.items())}")
    
print(Fore.CYAN + "sorted labels count for train_dataset:")
print(Style.RESET_ALL, end="")
for i in sorted(entity_counter.items()):
    print(i)
    

num_labels = 29
sorted labels count for train_dataset:
('BOD', 319698)
('BUILDING', 70435)
('CARDISSUER', 176)
('CITY', 182645)
('COUNTRY', 59025)
('DATE', 248231)
('DRIVERLICENSE', 457693)
('EMAIL', 563050)
('GEOCOORD', 48635)
('GIVENNAME1', 123999)
('GIVENNAME2', 32964)
('IDCARD', 342705)
('IP', 932781)
('LASTNAME1', 161895)
('LASTNAME2', 40951)
('LASTNAME3', 13210)
('O', 19633733)
('PASS', 272495)
('PASSPORT', 297523)
('POSTCODE', 123126)
('SECADDRESS', 51955)
('SEX', 94358)
('SOCIALNUMBER', 496335)
('STATE', 99797)
('STREET', 215847)
('TEL', 395481)
('TIME', 261316)
('TITLE', 79128)
('USERNAME', 388980)


In [ ]:
print(Fore.CYAN + "least common labels:")
for i in reversed(entity_counter.most_common()[-5:]):
    print(Style.RESET_ALL + str(i))

least common labels:
('CARDISSUER', 176)
('LASTNAME3', 13210)
('GIVENNAME2', 32964)
('LASTNAME2', 40951)
('GEOCOORD', 48635)


In [15]:
offsets = ex_tokenized_with_offsets["offset_mapping"]
example_aligned = align_tokens_to_labels(example, offsets, verbose=True) #type: ignore

NameError: name 'align_tokens_to_labels' is not defined

In [15]:
offsets = ex_tokenized_with_offsets["offset_mapping"]
example_aligned = get_ner_labels(example, offsets)

In [18]:
print(example_aligned)

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'USERNAME', 'USERNAME', 'USERNAME', 'USERNAME', 'USERNAME', 'USERNAME', 'USERNAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'TIME', 'TIME', 'TIME', 'TIME', 'O', 'O', 'O', 'O', 'O'

In [18]:
entity_classes = label_info["entity_classes"]
label2id = {"O": 0}
for i, entity in enumerate(entity_classes):
    b_id = 1 + i * 2
    i_id = b_id + 1
    entity_tags = {f"B-{entity}": b_id, f"I-{entity}": i_id}
    label2id.update(entity_tags)
print(f"{len(label2id)=}")
pprint(label2id, sort_dicts=False)

len(label2id)=57
{'O': 0,
 'B-BOD': 1,
 'I-BOD': 2,
 'B-BUILDING': 3,
 'I-BUILDING': 4,
 'B-CARDISSUER': 5,
 'I-CARDISSUER': 6,
 'B-CITY': 7,
 'I-CITY': 8,
 'B-COUNTRY': 9,
 'I-COUNTRY': 10,
 'B-DATE': 11,
 'I-DATE': 12,
 'B-DRIVERLICENSE': 13,
 'I-DRIVERLICENSE': 14,
 'B-EMAIL': 15,
 'I-EMAIL': 16,
 'B-GEOCOORD': 17,
 'I-GEOCOORD': 18,
 'B-GIVENNAME1': 19,
 'I-GIVENNAME1': 20,
 'B-GIVENNAME2': 21,
 'I-GIVENNAME2': 22,
 'B-IDCARD': 23,
 'I-IDCARD': 24,
 'B-IP': 25,
 'I-IP': 26,
 'B-LASTNAME1': 27,
 'I-LASTNAME1': 28,
 'B-LASTNAME2': 29,
 'I-LASTNAME2': 30,
 'B-LASTNAME3': 31,
 'I-LASTNAME3': 32,
 'B-PASS': 33,
 'I-PASS': 34,
 'B-PASSPORT': 35,
 'I-PASSPORT': 36,
 'B-POSTCODE': 37,
 'I-POSTCODE': 38,
 'B-SECADDRESS': 39,
 'I-SECADDRESS': 40,
 'B-SEX': 41,
 'I-SEX': 42,
 'B-SOCIALNUMBER': 43,
 'I-SOCIALNUMBER': 44,
 'B-STATE': 45,
 'I-STATE': 46,
 'B-STREET': 47,
 'I-STREET': 48,
 'B-TEL': 49,
 'I-TEL': 50,
 'B-TIME': 51,
 'I-TIME': 52,
 'B-TITLE': 53,
 'I-TITLE': 54,
 'B-USERNAME': 55,


In [19]:
def tokenize_and_align_labels(batch: dict[str, list[list]]):
    tokenized_inputs = tokenizer(
        batch["source_text"],
        truncation=True,
    )
    
    
    labels = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[tags[word_idx]])  
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [20]:
dataset = dataset.map(tokenize_and_align_labels, batched=True, batch_size=20_000)

Map:   0%|          | 0/177677 [00:00<?, ? examples/s]

Map:   0%|          | 0/47728 [00:00<?, ? examples/s]

In [21]:
pprint(dataset["train"].features)

{'attention_mask': List(Value('int8')),
 'input_ids': List(Value('int32')),
 'labels': List(Value('int64')),
 'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string'),
 'token_type_ids': List(Value('int8'))}


In [ ]:
ex = dataset["train"][0]
ex_wi = tokenizer(
    ex["source_text"],
    truncation=True,
).word_ids(batch_index=0)

id2label = {i: label for label, i in label2id.items()}
tokens = tokenizer.convert_ids_to_tokens(ex["input_ids"])
    
for label, wi in zip(ex["labels"], ex_wi):
    if label != -100 and label != 0:
        token = tokens[wi]
        value = f"label = {id2label[label]} word_id = {wi}, token = {token}"
        print(value)
    elif label == -100 and wi != None:
        token = tokens[wi]
        print(f"label = {label} word_id = {wi}, token = {token}")

In [24]:
print(f"{len(ex["ner_tags"])=}")
print(f"{len(ex["input_ids"])=}")

len(ex["ner_tags"])=117
len(ex["input_ids"])=119
